# HW06: Attention

Remember that these homework work as a completion grade. **You can <span style="color:red">not</span> skip one section this homework.**

### Data sampling with a sliding window

In this intro code I will ask you to revise what we saw on the lab to test your understanding of the features.

In [ ]:
#Import the AG news dataset (same as hw01)
#Download them from here
!wget https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv

import pandas as pd
import nltk
import torch
df = pd.read_csv('train.csv')

df.columns = ["label", "title", "lead"]
label_map = {1:"world", 2:"sport", 3:"business", 4:"sci/tech"}
def replace_label(x):
	return label_map[x]
df["label"] = df["label"].apply(replace_label)
df["text"] = df["title"] + " " + df["lead"]
df = df.sample(n=10000) # # only use 10K datapoints
df.head()

--2026-03-09 18:44:26--  https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 29470338 (28M) [text/plain]
Saving to: ‘train.csv’

train.csv           100%[===================>]  28.10M   171MB/s    in 0.2s    

2026-03-09 18:44:26 (171 MB/s) - ‘train.csv’ saved [29470338/29470338]



,label,title,lead,text
71705,business,"Wild Oats, Pathmark Warn on Results",Natural and organic foods retailer Wild Oats M...,"Wild Oats, Pathmark Warn on Results Natural an..."
107337,business,United union ballots members on strike proposal,WASHINGTON (AFP) - The union representing Unit...,United union ballots members on strike proposa...
89407,sci/tech,Virus warning: Cyborgs at risk,Humans will upgrade their nervous systems with...,Virus warning: Cyborgs at risk Humans will upg...
14719,sport,Masked intruder may have killed power to SkyDome,Power to the SkyDome and surrounding areas wen...,Masked intruder may have killed power to SkyDo...
59177,sci/tech,Motorola to trial m-commerce-enabled phones,Motorola Inc. will conduct an m-commerce field...,Motorola to trial m-commerce-enabled phones Mo...


In [ ]:
#TODO clean up the text as best as you can. Build a function to do this and generate a new variable called 'text_clean'
import pandas as pd
import re
import string
import nltk

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)

    return text

df['text_clean'] = df['text'].apply(clean_text)

df[['text', 'text_clean']].head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


,text,text_clean
71705,"Wild Oats, Pathmark Warn on Results Natural an...",wild oats pathmark warn on results natural an...
107337,United union ballots members on strike proposa...,united union ballots members on strike proposa...
89407,Virus warning: Cyborgs at risk Humans will upg...,virus warning cyborgs at risk humans will upg...
14719,Masked intruder may have killed power to SkyDo...,masked intruder may have killed power to skydo...
59177,Motorola to trial m-commerce-enabled phones Mo...,motorola to trial m commerce enabled phones mo...


In [ ]:
#TODO using byte-pair encoding, tokenize the first article of the dataset. select 10 tokens that make a coherent sentence
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

#

token_ids = tokenizer.encode(df.iloc[0]["text_clean"])

print("Token ids:", token_ids[10:30])
print("Decoded text:", tokenizer.decode(token_ids[10:30]))

Token ids: [10469, 9013, 21538, 4295, 47009, 5939, 753, 220, 220, 47009, 267, 220, 9577, 220, 7034, 220, 2267, 220, 290, 3108]
Decoded text:  organic foods retailer wild oats markets inc   oats o  quote  profile  research  and path


In [ ]:
#TODO For this 10 tokens, use the sliding window DataLoader seen in the lab to generate the input and target tensors:

from torch.utils.data import Dataset, DataLoader
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt) # Tokenizes the entire text

        for i in range(0, len(token_ids) - max_length, stride):  # Uses a sliding window to chunk the book into overlapping sequences of max_length
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self): # Returns the total number of rows in the dataset
        return len(self.input_ids)

    def __getitem__(self, idx):  # Returns a single row from the dataset
        return self.input_ids[idx], self.target_ids[idx]

# Now A data loader to generate batches with input-with pairs
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2") # Initializes the tokenizer
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride) # Creates dataset
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last, # drop_last=True drops the last batch if it is shorter than the specified batch_size to prevent loss spikes during training. (See Rashkca A6)
        num_workers=num_workers  # The number of CPU processes to use for preprocessing
    )

    return dataloader

comp = tokenizer.decode(token_ids[10:30])
dataloader = create_dataloader_v1(comp, batch_size=1,
                                  max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader) # Converts dataloader into a Python iterator to fetch the next entry
first_batch = next(data_iter) # via Python’s built-in next() function
print(first_batch) # Input and target tensors

second_batch = next(data_iter) # via Python’s built-in next() function
print(second_batch) # Input and target tensors

[tensor([[10469,  9013, 21538,  4295]]), tensor([[ 9013, 21538,  4295, 47009]])]
[tensor([[ 9013, 21538,  4295, 47009]]), tensor([[21538,  4295, 47009,  5939]])]


In [ ]:
dataloader = create_dataloader_v1(comp, batch_size=1,
                                  max_length=4, stride=2, shuffle=False)
data_iter = iter(dataloader) # Converts dataloader into a Python iterator to fetch the next entry
first_batch = next(data_iter) # via Python’s built-in next() function
print(first_batch) # Input and target tensors

second_batch = next(data_iter) # via Python’s built-in next() function
print(second_batch) # Input and target tensors

[tensor([[10469,  9013, 21538,  4295]]), tensor([[ 9013, 21538,  4295, 47009]])]
[tensor([[21538,  4295, 47009,  5939]]), tensor([[ 4295, 47009,  5939,   753]])]


In [ ]:
dataloader = create_dataloader_v1(comp, batch_size=1,
                                  max_length=10, stride=3, shuffle=False)
data_iter = iter(dataloader) # Converts dataloader into a Python iterator to fetch the next entry
first_batch = next(data_iter) # via Python’s built-in next() function
print(first_batch) # Input and target tensors

second_batch = next(data_iter) # via Python’s built-in next() function
print(second_batch) # Input and target tensors

third_batch = next(data_iter) # via Python’s built-in next() function
print(third_batch) # Input and target tensors

[tensor([[10469,  9013, 21538,  4295, 47009,  5939,   753,   220,   220, 47009]]), tensor([[ 9013, 21538,  4295, 47009,  5939,   753,   220,   220, 47009,   267]])]
[tensor([[ 4295, 47009,  5939,   753,   220,   220, 47009,   267,   220,  9577]]), tensor([[47009,  5939,   753,   220,   220, 47009,   267,   220,  9577,   220]])]
[tensor([[  753,   220,   220, 47009,   267,   220,  9577,   220,  7034,   220]]), tensor([[  220,   220, 47009,   267,   220,  9577,   220,  7034,   220,  2267]])]


Stride: How far to slide the window each iteration
Max_length: how much context to include for each key.

## Word and positional embeddings

In [ ]:
import torch
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

text = df.iloc[0]["text_clean"]

token_ids = tokenizer.encode(text)
print("Token IDs:", token_ids)
input_ids = torch.tensor(token_ids[:3])
print("Input token ids:", input_ids)
print("Decoded tokens:", [tokenizer.decode([i]) for i in input_ids.tolist()])

#TODO Using the input tokens, create a token embedding layer and look up the embeddings
vocab_size = tokenizer.n_vocab
embed_dim = 8

token_embedding_layer = torch.nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embed_dim
)

token_embeddings = token_embedding_layer(input_ids)

print("Token embedding shape:", token_embeddings.shape)
print("Token embeddings:\n", token_embeddings)
#TODO create a positional embedding layer
max_length = 10

pos_embedding_layer = torch.nn.Embedding(
    num_embeddings=max_length,
    embedding_dim=embed_dim
)
#TODO generate position indices [0, 1, 2] and look them up
position_indices = torch.arange(3)
print("Position indices:", position_indices)

position_embeddings = pos_embedding_layer(position_indices)

print("Position embedding shape:", position_embeddings.shape)
print("Position embeddings:\n", position_embeddings)
#TODO Sum them up and interpret the result (.shape, what are the values?)
input_embeddings = token_embeddings + position_embeddings

print("Summed embedding shape:", input_embeddings.shape)
print("Summed embeddings:\n", input_embeddings)

Token IDs: [21992, 47009, 220, 3108, 4102, 9828, 319, 2482, 3288, 290, 10469, 9013, 21538, 4295, 47009, 5939, 753, 220, 220, 47009, 267, 220, 9577, 220, 7034, 220, 2267, 220, 290, 3108, 4102, 7000, 753, 220, 220, 279, 17209, 74, 267, 220, 9577, 220, 7034, 220, 2267, 220, 1111, 7728, 319, 285, 3204, 326, 511, 220]
Input token ids: tensor([21992, 47009,   220])
Decoded tokens: ['wild', ' oats', ' ']
Token embedding shape: torch.Size([3, 8])
Token embeddings:
 tensor([[ 0.3075,  0.2331, -0.9171,  1.0094, -1.1414,  1.0907,  0.9386,  0.8424],
        [-0.7296, -1.3472, -0.3892,  0.8901, -1.5368, -0.4584,  1.2957,  0.9921],
        [ 0.8702, -0.1329,  0.9124,  0.4446, -0.4187, -0.6854, -0.1196,  0.5093]],
       grad_fn=<EmbeddingBackward0>)
Position indices: tensor([0, 1, 2])
Position embedding shape: torch.Size([3, 8])
Position embeddings:
 tensor([[ 0.8769,  0.5878, -1.2612, -0.2134,  0.2159,  0.6499,  0.9426,  2.0881],
        [-0.3400,  1.1773, -1.4017, -2.5347, -2.2156,  0.3895, -1.046

BERT I will go over that after Thursday